<a href="https://colab.research.google.com/github/guilhermef2k/Sistemas_Inteligentes/blob/main/Adaline/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4º Projeto - Sistemas Inteligentes

**Rede Adaline aplicada ao dataset Ionosphere**  
Autores: Alex Bruno Duarte e Guilherme de França Vasconcelos

## 1. Introdução
---

A ionosfera é uma camada da atmosfera terrestre fortemente ionizada pela radiação solar. O monitoramento de retornos de radar permite identificar se existe uma estrutura ionosférica coerente. Neste projeto, a tarefa é classificar os retornos como **good** (`+1`) ou **bad** (`-1`) utilizando uma rede **Adaline**.

## 2. Objetivo
---

Implementar uma Rede Adaline para classificar automaticamente os retornos de radar do dataset Ionosphere, avaliar diferentes taxas de aprendizagem e analisar o desempenho final por meio de matriz de confusão e métricas de classificação.

## 3. Modelo matemático
---

A rede Adaline combina linearmente as entradas:

$$z = w^T x + b$$

Durante o treinamento, a saída linear $z$ é comparada com o rótulo real $y \in \{-1, +1\}$. A função de perda utilizada é o erro quadrático médio:

$$J(w) = rac{1}{2n}\sum_{i=1}^{n}(y_i - z_i)^2$$

A atualização dos pesos segue a regra Delta, também conhecida como regra de Widrow-Hoff:

$$w \leftarrow w + \eta rac{1}{n}X^T(y-z)$$

$$b \leftarrow b + \etarac{1}{n}\sum_{i=1}^{n}(y_i-z_i)$$

## 4. Importação das bibliotecas

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

## 5. Carregamento do dataset
---

O arquivo `ionosphere.data` não possui cabeçalho. As 34 primeiras colunas são atributos preditores e a 35ª coluna é o rótulo da classe, com valores `g` (*good*) e `b` (*bad*).

In [ ]:
url = "https://raw.githubusercontent.com/guilhermef2k/Sistemas_Inteligentes/main/Adaline/dataSet/ionosphere.data"

colunas = [f"x{i}" for i in range(1, 35)] + ["classe"]
dados = pd.read_csv(url, header=None, names=colunas)

dados.head()

## 6. Análise inicial dos dados

In [ ]:
print("Dimensão do dataset:", dados.shape)
print("\nDistribuição original das classes:")
print(dados["classe"].value_counts())

print("\nQuantidade de valores ausentes por coluna:")
print(dados.isnull().sum())

## 7. Recodificação da classe
---

O Adaline será treinado no padrão bipolar:

* `g` = `+1`
* `b` = `-1`

In [ ]:
dados["classe"] = dados["classe"].map({"g": 1, "b": -1})

print(dados["classe"].value_counts())

## 8. Separação entre atributos e alvo

In [ ]:
X = dados.drop(columns=["classe"])
y = dados["classe"].values

X.head()

## 9. Verificação e remoção de atributos constantes
---

Atributos constantes não contribuem para a classificação, pois possuem variância zero. Além disso, na padronização Z-score, uma variância nula causaria divisão por zero.

In [ ]:
variancias = X.var(axis=0)
atributos_constantes = variancias[variancias == 0].index.tolist()

print("Atributos constantes encontrados:", atributos_constantes)

X = X.drop(columns=atributos_constantes)

print("Nova dimensão de X:", X.shape)

## 10. Divisão treino/teste
---

A divisão será feita com 80% para treino e 20% para teste. O parâmetro `stratify=y` mantém a proporção das classes em ambos os conjuntos.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)
print("\nDistribuição no treino:")
print(pd.Series(y_train).value_counts())
print("\nDistribuição no teste:")
print(pd.Series(y_test).value_counts())

## 11. Padronização dos atributos
---

A normalização é vital para a convergência do Gradiente Descendente. Como a atualização dos pesos depende diretamente da magnitude dos atributos, variáveis em escalas maiores podem dominar o ajuste e tornar a descida instável.

A padronização Z-score transforma cada atributo em:

$$x^{\prime}=rac{x-\mu}{\sigma}$$

Assim, os atributos passam a ter média próxima de zero e desvio padrão igual a um, deixando a superfície da função de perda mais equilibrada para o Adaline.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 12. Implementação manual da Rede Adaline
---

A implementação abaixo usa a regra Delta/Widrow-Hoff com atualização em lote. A saída usada no treinamento é linear; apenas na predição final aplica-se a função degrau bipolar.

In [ ]:
class AdalineGD:
    def __init__(self, eta=0.01, n_iter=200, random_state=42):
        self.eta = eta
        self.n_iter = n_iter
        self.random_state = random_state

    def fit(self, X, y):
        rng = np.random.default_rng(self.random_state)
        self.w_ = rng.normal(loc=0.0, scale=0.01, size=X.shape[1])
        self.b_ = 0.0
        self.losses_ = []

        for _ in range(self.n_iter):
            net_input = self.net_input(X)
            errors = y - net_input

            self.w_ += self.eta * X.T.dot(errors) / X.shape[0]
            self.b_ += self.eta * errors.mean()

            loss = (errors ** 2).mean() / 2.0
            self.losses_.append(loss)

        return self

    def net_input(self, X):
        return np.dot(X, self.w_) + self.b_

    def predict(self, X):
        return np.where(self.net_input(X) >= 0.0, 1, -1)

## 13. Estudo da taxa de aprendizagem

In [ ]:
etas = [1e-4, 1e-3, 1e-2, 5e-2, 1e-1]
modelos = {}
resultados_eta = []

for eta in etas:
    modelo = AdalineGD(eta=eta, n_iter=200, random_state=42)
    modelo.fit(X_train_scaled, y_train)
    modelos[eta] = modelo

    resultados_eta.append({
        "eta": eta,
        "loss_final": modelo.losses_[-1],
        "loss_minima": np.min(modelo.losses_)
    })

resultados_eta = pd.DataFrame(resultados_eta)
resultados_eta

## 14. Curvas de convergência

In [ ]:
plt.figure(figsize=(10, 6))

for eta, modelo in modelos.items():
    plt.plot(range(1, len(modelo.losses_) + 1), modelo.losses_, label=f"eta = {eta}")

plt.xlabel("Épocas")
plt.ylabel("Função de perda")
plt.title("Curvas de convergência do Adaline para diferentes taxas de aprendizagem")
plt.legend()
plt.grid(True)
plt.show()

## 15. Escolha automática do melhor eta
---

O melhor valor será escolhido pelo menor valor final da função de perda. Caso duas taxas tenham perdas próximas, recomenda-se escolher a curva mais suave, pois ela indica uma convergência mais estável.

In [ ]:
melhor_eta = resultados_eta.sort_values(by="loss_final").iloc[0]["eta"]

print("Melhor eta encontrado:", melhor_eta)
resultados_eta.sort_values(by="loss_final")

## 16. Treinamento final com o melhor eta

In [ ]:
modelo_final = AdalineGD(eta=melhor_eta, n_iter=200, random_state=42)
modelo_final.fit(X_train_scaled, y_train)

y_pred = modelo_final.predict(X_test_scaled)

## 17. Matriz de confusão

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[1, -1])

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["good (+1)", "bad (-1)"]
)

disp.plot(values_format="d")
plt.title("Matriz de Confusão - Adaline")
plt.show()

## 18. Métricas de avaliação

In [ ]:
acuracia = accuracy_score(y_test, y_pred)
precisao = precision_score(y_test, y_pred, pos_label=1)
recall = recall_score(y_test, y_pred, pos_label=1)
f1 = f1_score(y_test, y_pred, pos_label=1)

print(f"Acurácia: {acuracia:.4f}")
print(f"Precisão: {precisao:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")

print("\nRelatório de classificação:")
print(classification_report(
    y_test,
    y_pred,
    labels=[1, -1],
    target_names=["good (+1)", "bad (-1)"]
))

## 19. Análise crítica dos erros
---

Em um sistema real de monitoramento automático, o erro mais crítico é classificar um sinal **bad** como **good**.

Isso significa que o sistema interpretaria um retorno ruim, sem estrutura ionosférica coerente, como se fosse um sinal confiável. Em aplicações de telecomunicações, GPS ou monitoramento geomagnético, esse falso positivo pode mascarar falhas de transmissão, interferências ou ausência de estrutura detectável.

O erro oposto, classificar um sinal **good** como **bad**, também é indesejável, pois pode gerar alarmes falsos. Porém, do ponto de vista operacional, é geralmente mais seguro disparar uma verificação desnecessária do que aceitar como confiável um sinal que deveria ser tratado como problemático.

## 20. Conclusão
---

A Rede Adaline apresentou comportamento sensível à taxa de aprendizagem. Valores muito baixos de $\eta$ tendem a convergir lentamente, enquanto valores muito altos podem causar oscilações ou instabilidade na função de perda. A padronização dos atributos foi essencial para deixar o treinamento numericamente mais estável.

Com o melhor valor de $\eta$ identificado pelas curvas de convergência, o modelo final foi avaliado no conjunto de teste por meio da matriz de confusão, acurácia, precisão, recall e F1-score. O Adaline é adequado como modelo linear de base para este problema, embora modelos não lineares possam capturar relações mais complexas do dataset Ionosphere.